# Checking what kind of data the geodatabase has
Land cover data provided by PRRIP are in a gdb format that can be accessed through ArcGIS, not through ArcGIS Pro.
We are using a Python script to understand the data that is inside gdb. We performed the following steps to understand the data.
1. Check what type of data is in the geodatabase
2. Export land cover from the geodatabase
3. Inspect the attribute table in land cover
4. Remap the land cover classification into a smaller number and then resample into a 30 m resolution
5. The next step is to align the resampled landcover with the Landsat image, which will be performed in HCC during model training
First attempt to see if GBD has a shapefile or vector data

In [1]:
# Cell A — point to your GDB (raw string for spaces)
# Change this file path to the location where geodatabase is located in your working directly
GDB_PATH = r"E:\Land Cover\combinedLandcover\combinedLandcover.gdb" 
import os, pathlib
p = pathlib.Path(GDB_PATH)
print("Exists?      ", p.exists())
print("Is dir?      ", p.is_dir())
print("Drive RW?    ", os.access(p.drive + "\\", os.R_OK))
print("Folder RW?   ", os.access(str(p), os.R_OK))

# List a few files in the .gdb folder
for i, f in enumerate(p.iterdir()):
    print(" -", f.name)
    if i > 15: break


Exists?       True
Is dir?       True
Drive RW?     True
Folder RW?    True
 - a00000001.freelist
 - a00000001.gdbindexes
 - a00000001.gdbtable
 - a00000001.gdbtablx
 - a00000001.TablesByName.atx
 - a00000002.gdbtable
 - a00000002.gdbtablx
 - a00000003.gdbindexes
 - a00000003.gdbtable
 - a00000003.gdbtablx
 - a00000004.CatItemsByPhysicalName.atx
 - a00000004.CatItemsByType.atx
 - a00000004.FDO_UUID.atx
 - a00000004.freelist
 - a00000004.gdbindexes
 - a00000004.gdbtable
 - a00000004.gdbtablx


In [ ]:

import os, shutil, pathlib, sys, traceback
import pandas as pd

try:
    import pyogrio
    HAVE_PYOGRIO = True
except Exception:
    HAVE_PYOGRIO = False

import fiona

# EDIT THIS to the *inner* .gdb that contains a0000000*.gdb* files
GDB_PATH = r"E:\Land Cover\combinedLandcover\combinedLandcover.gdb"

LOCAL_COPY = r"C:\gdb\combined.gdb"  # short local fallback path (will be created if needed)

def ensure_real_gdb(path: str) -> str:
    """Return a path to a folder that contains .gdbtable files."""
    p = pathlib.Path(path)
    if not p.exists() or not p.is_dir():
        raise FileNotFoundError(path)
    # already real gdb?
    if any(ch.suffix == ".gdbtable" for ch in p.iterdir() if ch.is_file()):
        return str(p)
    # search down one/two levels
    for root, dirs, files in os.walk(str(p)):
        for d in dirs:
            cand = pathlib.Path(root) / d
            if cand.name.lower().endswith(".gdb"):
                if any(ch.suffix == ".gdbtable" for ch in cand.iterdir() if ch.is_file()):
                    return str(cand)
    raise RuntimeError("Could not find a real .gdb folder with .gdbtable files under: " + path)

def to_longpath(p: str) -> str:
    """Windows long-path prefix; safe to pass to GDAL/pyogrio."""
    if os.name == "nt":
        p = os.path.normpath(p)
        if not p.startswith("\\\\?\\"):
            return "\\\\?\\" + p
    return p

def list_layers_all_ways(gdb: str):
    """Try several ways to list layers; returns list of dicts."""
    attempts = []
    # Preferred URIs for forcing driver:
    uri_forward = "OpenFileGDB:" + gdb.replace("\\", "/")
    uri_long    = "OpenFileGDB:" + to_longpath(gdb).replace("\\", "/")

    # 1) pyogrio with URI
    if HAVE_PYOGRIO:
        try:
            L = pyogrio.list_layers(uri_forward)
            return [{"layer": n, "geometry": g, "crs_present": bool(srs)} for n, g, srs in L]
        except Exception as e:
            attempts.append(("pyogrio uri_forward", str(e)))
        try:
            L = pyogrio.list_layers(uri_long)
            return [{"layer": n, "geometry": g, "crs_present": bool(srs)} for n, g, srs in L]
        except Exception as e:
            attempts.append(("pyogrio uri_long", str(e)))
        # fall back to direct path
        try:
            L = pyogrio.list_layers(gdb)
            return [{"layer": n, "geometry": g, "crs_present": bool(srs)} for n, g, srs in L]
        except Exception as e:
            attempts.append(("pyogrio direct", str(e)))

    # 2) Fiona with URI
    try:
        L = fiona.listlayers(uri_forward)
        out = []
        for lyr in L:
            try:
                with fiona.open(uri_forward, layer=lyr) as src:
                    out.append({"layer": lyr,
                                "geometry": src.schema.get("geometry") or "",
                                "crs_present": bool(src.crs)})
            except Exception:
                out.append({"layer": lyr, "geometry": "", "crs_present": False})
        return out
    except Exception as e:
        attempts.append(("fiona uri_forward", str(e)))
    try:
        L = fiona.listlayers(uri_long)
        out = []
        for lyr in L:
            try:
                with fiona.open(uri_long, layer=lyr) as src:
                    out.append({"layer": lyr,
                                "geometry": src.schema.get("geometry") or "",
                                "crs_present": bool(src.crs)})
            except Exception:
                out.append({"layer": lyr, "geometry": "", "crs_present": False})
        return out
    except Exception as e:
        attempts.append(("fiona uri_long", str(e)))
    # 3) Fiona direct path
    try:
        L = fiona.listlayers(gdb)
        out = []
        for lyr in L:
            try:
                with fiona.open(gdb, layer=lyr) as src:
                    out.append({"layer": lyr,
                                "geometry": src.schema.get("geometry") or "",
                                "crs_present": bool(src.crs)})
            except Exception:
                out.append({"layer": lyr, "geometry": "", "crs_present": False})
        return out
    except Exception as e:
        attempts.append(("fiona direct", str(e)))
        raise RuntimeError("All layer-list attempts failed:\n" + "\n".join(f"- {k}: {v}" for k, v in attempts))


In [ ]:
# checking if they tiff file

In [ ]:

REAL_GDB = ensure_real_gdb(GDB_PATH)
print("Using GDB:", REAL_GDB)

try:
    layers_info = list_layers_all_ways(REAL_GDB)
    inv = pd.DataFrame(layers_info).assign(
        dataset_type=lambda d: d["geometry"].apply(lambda g: "FeatureClass" if g else "Table")
    ).sort_values(["dataset_type","layer"]).reset_index(drop=True)
    display(inv)
except Exception as e:
    print("Listing failed:", e)


In [ ]:
import rasterio
import os

GDB_PATH = r"E:\Land Cover\combinedLandcover\combinedLandcover.gdb"

# rasterio expects a dataset path inside the GDB, so we can just try opening the folder
try:
    with rasterio.open(GDB_PATH) as src:
        print("Raster opened successfully!")
        print("Driver:", src.driver)
        print("Size:", src.width, "x", src.height)
        print("Bands:", src.count)
        print("CRS:", src.crs)
except Exception as e:
    print("Not a single raster dataset. Error:", e)


# checking data from subfolder inside geodatabase

In [1]:
from osgeo import gdal

GDB_PATH = r"E:\Land Cover\combinedLandcover\combinedLandcover.gdb"

ds = gdal.Open(GDB_PATH)
if ds is None:
    print("Could not open dataset.")
else:
    print("Subdatasets:")
    for i, s in enumerate(ds.GetSubDatasets()):
        print(i, s[0])


C:\Users\schaulagain4\.conda\envs\gdbtools\Lib\site-packages\osgeo\gdal.py:330: FutureWarning: Neither gdal.UseExceptions() nor gdal.DontUseExceptions() has been explicitly called. In GDAL 4.0, exceptions will be enabled by default.
  warnings.warn(


Subdatasets:
0 OpenFileGDB:"E:\Land Cover\combinedLandcover\combinedLandcover.gdb":combinedLandcover00_02
1 OpenFileGDB:"E:\Land Cover\combinedLandcover\combinedLandcover.gdb":combinedLandcover2011
2 OpenFileGDB:"E:\Land Cover\combinedLandcover\combinedLandcover.gdb":combinedLandcover2016
3 OpenFileGDB:"E:\Land Cover\combinedLandcover\combinedLandcover.gdb":combinedLandcover2006
4 OpenFileGDB:"E:\Land Cover\combinedLandcover\combinedLandcover.gdb":combinedLandcover2010
5 OpenFileGDB:"E:\Land Cover\combinedLandcover\combinedLandcover.gdb":combinedLandcover2013
6 OpenFileGDB:"E:\Land Cover\combinedLandcover\combinedLandcover.gdb":combinedLandcover2015
7 OpenFileGDB:"E:\Land Cover\combinedLandcover\combinedLandcover.gdb":combinedLandcover2014
8 OpenFileGDB:"E:\Land Cover\combinedLandcover\combinedLandcover.gdb":combinedLandcover2017
9 OpenFileGDB:"E:\Land Cover\combinedLandcover\combinedLandcover.gdb":combinedLandcover2009
10 OpenFileGDB:"E:\Land Cover\combinedLandcover\combinedLandcover.

# Checking existing attribute table of tiff file

In [1]:
from osgeo import gdal

# Path to subdataset
subdataset_path = r"OpenFileGDB:E:\Land Cover\combinedLandcover\combinedLandcover.gdb:combinedLandcover2020"

ds = gdal.Open(subdataset_path)
band = ds.GetRasterBand(1)

rat = band.GetDefaultRAT()
if rat is None:
    print("No Raster Attribute Table found.")
else:
    print("RAT columns:", [rat.GetNameOfCol(i) for i in range(rat.GetColumnCount())])
    for i in range(rat.GetRowCount()):
        row = [rat.GetValueAsString(i, j) for j in range(rat.GetColumnCount())]
        print(row)


C:\Users\schaulagain4\.conda\envs\gdbtools\Lib\site-packages\osgeo\gdal.py:330: FutureWarning: Neither gdal.UseExceptions() nor gdal.DontUseExceptions() has been explicitly called. In GDAL 4.0, exceptions will be enabled by default.
  warnings.warn(


RAT columns: ['Value', 'Count', 'lcClass', 'lcPropClass']
['0', '1986469', '', '']
['1', '1114247', 'Upland Agriculture', '']
['2', '6413090', 'Bareground/Sparse Veg', '']
['3', '625758', 'Canal/Drainage', '']
['4', '71003', 'Irrigation Reuse Pit', '']
['5', '177462', 'Lagoon', '']
['6', '6214083', 'Meadow Sand Ridge', 'grassland']
['7', '39577656', 'Mesic Wet Meadow', 'meadowMarsh']
['8', '3858938', 'Phragmites', '']
['9', '218103', 'Purple Loosestrife', '']
['10', '104657837', 'Riparian Woodland', 'forest']
['11', '29715834', 'Roads', 'developed']
['12', '54107431', 'Rural Development', 'forest']
['13', '317175', 'Stock Pond', '']
['14', '1758463', 'Undisturbed Grassland', 'grassland']
['15', '16900234', 'Upland Grassland', 'grassland']
['16', '1329427', 'Upland Shrubland', '']
['17', '11279598', 'Upland Woodland', 'forest']
['18', '4009123', 'Urban/Suburban', 'developed']
['19', '133956929', 'Xeric Wet Meadow', 'grassland']
['20', '47771', 'Floodplain Marsh', 'meadowMarsh']
['21', '

In [ ]:
# Exporting tif image from geodatabase

In [1]:
import os
from osgeo import gdal


# Paths
GDB_PATH = r"E:\Land Cover\combinedLandcover\combinedLandcover.gdb"
OUT_DIR = r"E:\Land Cover\exports"
os.makedirs(OUT_DIR, exist_ok=True)

# Years to export
years = [2003, 2004, 2006, 2007, 2008, 2009, 2011, 2012, 2013, 2014, 2016, 2018, 2019, 2021]

for year in years:
    subdataset_name = f"combinedLandcover{year}"
    subdataset_path = f"OpenFileGDB:{GDB_PATH}:{subdataset_name}"
    out_tif = os.path.join(OUT_DIR, f"{subdataset_name}.tif")

    print(f"Exporting {subdataset_name} ...")
    gdal.Translate(out_tif, subdataset_path)
    print(f"Saved to {out_tif}")

print("✅ Export finished.")


Exporting combinedLandcover2003 ...


C:\Users\schaulagain4\.conda\envs\gdbtools\Lib\site-packages\osgeo\gdal.py:330: FutureWarning: Neither gdal.UseExceptions() nor gdal.DontUseExceptions() has been explicitly called. In GDAL 4.0, exceptions will be enabled by default.
  warnings.warn(


Saved to E:\Land Cover\exports\combinedLandcover2003.tif
Exporting combinedLandcover2004 ...
Saved to E:\Land Cover\exports\combinedLandcover2004.tif
Exporting combinedLandcover2006 ...
Saved to E:\Land Cover\exports\combinedLandcover2006.tif
Exporting combinedLandcover2007 ...
Saved to E:\Land Cover\exports\combinedLandcover2007.tif
Exporting combinedLandcover2008 ...
Saved to E:\Land Cover\exports\combinedLandcover2008.tif
Exporting combinedLandcover2009 ...
Saved to E:\Land Cover\exports\combinedLandcover2009.tif
Exporting combinedLandcover2011 ...
Saved to E:\Land Cover\exports\combinedLandcover2011.tif
Exporting combinedLandcover2012 ...
Saved to E:\Land Cover\exports\combinedLandcover2012.tif
Exporting combinedLandcover2013 ...
Saved to E:\Land Cover\exports\combinedLandcover2013.tif
Exporting combinedLandcover2014 ...
Saved to E:\Land Cover\exports\combinedLandcover2014.tif
Exporting combinedLandcover2016 ...
Saved to E:\Land Cover\exports\combinedLandcover2016.tif
Exporting com

In [ ]:
# exporting one tiff file that does not have date at the end of the name

In [ ]:
from osgeo import gdal
import os

GDB_PATH = r"E:\Land Cover\combinedLandcover\combinedLandcover.gdb"
OUT_DIR  = r"E:\Land Cover\exports"
os.makedirs(OUT_DIR, exist_ok=True)

layer = "combinedLandcover00_02"  # exactly as listed

# Quote the GDB path because it has spaces
src = f'OpenFileGDB:"{GDB_PATH}":{layer}'
dst = os.path.join(OUT_DIR, f"{layer}.tif")

print("Source:", src)
print("Dest  :", dst)

ds = gdal.Translate(
    dst,
    src,
    creationOptions=["COMPRESS=LZW", "TILED=YES"]
)

if ds is None:
    # show GDAL’s last error for easier debugging
    print("❌ gdal.Translate failed")
    err = gdal.GetLastErrorMsg()
    if err:
        print("GDAL says:", err)
else:
    ds = None
    print("✅ Saved:", dst)


# combining multiple Landsat image tiles exported from Google Earth Engine
Due to the large study area (area of interest), the exported Landsat image is in multiple tiles

In [19]:

from pathlib import Path
import re
import rasterio
from rasterio.merge import merge
import shutil

LANDSAT_DIR = Path(r"C:\Smriti_Project\PlatteRiver\PlatteRiver\Landsat\drive-download-20251104T123235Z-1-001")  # <- your folder

def mosaic_rasters(in_paths, out_path):
    srcs = [rasterio.open(p) for p in in_paths]
    try:
        crs0, cnt0, dt0, nd0 = srcs[0].crs, srcs[0].count, srcs[0].dtypes[0], srcs[0].nodata
        for s in srcs[1:]:
            assert s.crs == crs0, "CRS mismatch"
            assert s.count == cnt0, "Band count mismatch"
            assert s.dtypes[0] == dt0, "Dtype mismatch"
        mosaic, out_trans = merge(srcs, nodata=nd0)  # stitch
        meta = srcs[0].meta.copy()
        meta.update(
            height=mosaic.shape[1],
            width=mosaic.shape[2],
            transform=out_trans,
            driver="GTiff",
            compress="lzw",
            tiled=True,
            nodata=nd0
        )
        out_path.parent.mkdir(parents=True, exist_ok=True)
        with rasterio.open(out_path, "w", **meta) as dst:
            dst.write(mosaic)
        print(f"Saved {out_path}")
    finally:
        for s in srcs:
            s.close()

# Group tiles by year prefix like "LandsatStack_2015-"
year_groups = {}
for tif in LANDSAT_DIR.glob("*.tif"):
    if tif.suffix.lower() != ".tif": 
        continue
    if tif.name.endswith(".ovr.tif"):  # super cautious
        continue
    if tif.name.endswith(".ovr"):      # ignore overview files
        continue
    m = re.match(r"(LandsatStack_(\d{4}))-[0-9]+-[0-9]+\.tif$", tif.name)
    if not m:
        # if it’s already a full file, skip grouping here
        continue
    key = m.group(1)  # e.g., "LandsatStack_2015"
    year_groups.setdefault(key, []).append(tif)

for key, files in year_groups.items():
    year = key.split("_")[1]
    out_full = LANDSAT_DIR / f"{key}_full.tif"
    if out_full.exists():
        print(f"Already exists, skipping: {out_full}")
        continue
    files_sorted = sorted(files)  # names encode tile offsets; sort is fine
    if len(files_sorted) == 1:
        # only one tile—just copy as full
        shutil.copy2(files_sorted[0], out_full)
        print(f"Single tile -> copied to {out_full}")
    else:
        mosaic_rasters(files_sorted, out_full)


Already exists, skipping: C:\Smriti_Project\PlatteRiver\PlatteRiver\Landsat\drive-download-20251104T123235Z-1-001\LandsatStack_1985_full.tif
Already exists, skipping: C:\Smriti_Project\PlatteRiver\PlatteRiver\Landsat\drive-download-20251104T123235Z-1-001\LandsatStack_1990_full.tif
Saved C:\Smriti_Project\PlatteRiver\PlatteRiver\Landsat\drive-download-20251104T123235Z-1-001\LandsatStack_1995_full.tif
Saved C:\Smriti_Project\PlatteRiver\PlatteRiver\Landsat\drive-download-20251104T123235Z-1-001\LandsatStack_2000_full.tif
Saved C:\Smriti_Project\PlatteRiver\PlatteRiver\Landsat\drive-download-20251104T123235Z-1-001\LandsatStack_2005_full.tif
Saved C:\Smriti_Project\PlatteRiver\PlatteRiver\Landsat\drive-download-20251104T123235Z-1-001\LandsatStack_2010_full.tif
Saved C:\Smriti_Project\PlatteRiver\PlatteRiver\Landsat\drive-download-20251104T123235Z-1-001\LandsatStack_2015_full.tif
Saved C:\Smriti_Project\PlatteRiver\PlatteRiver\Landsat\drive-download-20251104T123235Z-1-001\LandsatStack_2020_f

# Exporting land cover labels one at a time
In this step, we are only remapping and resampling the land cover, since it takes longer without a GPU. In this step we bascially creating the labels that will be used to train the CNN UNet model.

In [3]:
#!/usr/bin/env python
# -*- coding: utf-8 -*-
"""
Create a compact 30 m, 5-class label raster (remap -> downsample with mode)
in the HR raster's UTM CRS. Later, align to Landsat on HCC with gdalwarp.

Input:
  HR_IN   = E:\Land Cover\exports\combinedLandcover2015.tif  (1 m categorical codes)
Output:
  OUT_30M = E:\Land Cover\labels_30m\2015\LabelHard_5cls_30m_srcgrid.tif (uint8)

Classes:
  1=Crops, 2=RiparianVeg, 3=Wetlands, 4=Sand/Land, 5=Other; nodata=255
"""

from pathlib import Path
import math
import numpy as np
import rasterio
from rasterio.enums import Resampling
from rasterio.transform import from_origin
from rasterio.vrt import WarpedVRT

# ---------------- CONFIG: EDIT THESE ----------------
BASE      = Path(r"E:\Land Cover")
HR_IN     = BASE / "exports" / "combinedLandcover2005.tif"     # your 1 m input
OUT_30M   = BASE / "labels_30m" / "2020" / "LabelHard_5cls_30m_2005.tif"
OUT_30M.parent.mkdir(parents=True, exist_ok=True)

TARGET_RES = 30.0   # meters (OK because HR CRS is UTM)
OUT_NODATA = 255    # output nodata that won't collide with 1..5 labels

# Remap table: detailed codes -> 5 classes

# 1=Water, 2=RiparianVeg, 3=SandLand, 4=Crops, 5=Other (Developed)
CLASS_MAP = {
    # --- 1: Water ---
    3:1,     # Canal/Drainage
    4:1,     # Irrigation Reuse Pit
    5:1,     # Lagoon
    13:1,    # Stock Pond
    6000:1,  # Water

    # --- 2: Riparian Vegetation (natural veg near streams, meadows, shrubs, etc.) ---
    6:2, 8:2, 9:2, 10:2, 14:2, 15:2, 16:2, 17:2, 21:2, 23:2, 24:2,
    100:2, 2000:2, 3000:2, 4000:2, 5000:2,  # veg height and transitional
    29:2, 30:2,                             # riparian/wet prairie edges

    # --- 3: Sand / Land (bare ground, channel, clearing, sandbars, pits) ---
    2:3, 22:3, 25:3, 26:3, 1000:3, 10000:3, 20000:3,

    # --- 4: Crops ---
    1:4, 200:4, 300:4, 400:4, 500:4, 600:4,

    # --- 5: Wetlands (marshes, sloughs, wet meadows, NWI wetlands) ---
    7:5, 19:5, 20:5, 27:5, 28:5, 29:5, 30:5,

    # --- 6: Other / Developed ---
    0:6, 11:6, 12:6, 18:6, 700:6
}


RESAMPLE_METHOD = getattr(Resampling, "mode", Resampling.nearest)

def snap_down(v, step):  return math.floor(v / step) * step
def snap_up(v, step):    return math.ceil(v / step) * step

def main():
    # 1) Stream-remap the HR input to a small 1..5 raster (in place of making a temp file)
    #    We'll remap on-the-fly while resampling; to keep RAM tiny we’ll do:
    #    - read HR in VRT windows after remap? Not directly—so we build a small helper:
    #    Approach: create a virtual 30m grid WarpedVRT and feed it blocks from a remap function.
    #    Since WarpedVRT reads from src, we’ll remap BEFORE resampling by writing a VRT over a remapped reader.
    #    Pure-rasterio doesn’t allow a custom per-pixel function in VRT, so we remap first to a virtual 1..5 file:
    #    => Instead, we stream-write a REMAP file blockwise, then VRT-resample that. (still one final file kept)

    # To avoid keeping an extra big file, we’ll write the remap to a temporary on-disk COG-like GTiff that
    # compresses well (uint8 + LZW), then immediately downsample it to 30 m, and delete the remap file.
    # (This is usually much smaller than your original HR input.)

    tmp_remap = OUT_30M.with_name("tmp_remap_5cls_uint8.tif")

    with rasterio.open(HR_IN) as src:
        src_profile = src.profile
        src_nodata = src.nodata
        if src_nodata is None:
            # choose an input nodata that doesn't collide with codes; 0 is fine if code 0 maps to class 5 anyway
            src_nodata = 0

        # Prepare REMAP output (same grid/CRS as HR input)
        remap_prof = src_profile.copy()
        remap_prof.update(
            driver="GTiff", dtype="uint8", count=1,
            nodata=OUT_NODATA, compress="lzw", tiled=True,
            blockxsize=256, blockysize=256
        )

        with rasterio.open(tmp_remap, "w", **remap_prof) as dst:
            for ji, win in src.block_windows(1):
                block = src.read(1, window=win)
                out = np.full(block.shape, OUT_NODATA, np.uint8)
                # preserve nodata
                out[block == src_nodata] = OUT_NODATA
                # remap known codes -> 1..5
                for k, v in CLASS_MAP.items():
                    out[block == k] = np.uint8(v)
                dst.write(out, 1, window=win)

    # 2) Build a 30 m grid in the same CRS as HR (snapped to 30 m lattice), and resample REMAP via mode
    with rasterio.open(HR_IN) as src:
        xmin, ymin, xmax, ymax = src.bounds
        dst_crs = src.crs

    xmin_s = snap_down(xmin, TARGET_RES)
    ymin_s = snap_down(ymin, TARGET_RES)
    xmax_s = snap_up(xmax, TARGET_RES)
    ymax_s = snap_up(ymax, TARGET_RES)

    width  = int(round((xmax_s - xmin_s) / TARGET_RES))
    height = int(round((ymax_s - ymin_s) / TARGET_RES))
    dst_transform = from_origin(xmin_s, ymax_s, TARGET_RES, TARGET_RES)

    with rasterio.open(tmp_remap) as src5:
        vrt_opts = dict(
            crs=dst_crs,
            transform=dst_transform,
            width=width,
            height=height,
            resampling=RESAMPLE_METHOD,
            src_nodata=OUT_NODATA
        )
        with WarpedVRT(src5, **vrt_opts) as vrt:
            prof = vrt.profile
            prof.update(
                driver="GTiff",
                dtype="uint8",
                count=1,
                nodata=OUT_NODATA,
                compress="lzw",
                tiled=True,
                blockxsize=256,
                blockysize=256
            )
            with rasterio.open(OUT_30M, "w", **prof) as dst:
                for ji, win in dst.block_windows(1):
                    data = vrt.read(1, window=win)
                    # ensure nodata is 255
                    data = data.astype("uint8", copy=False)
                    dst.write(data, 1, window=win)

    # 3) Clean up intermediate remap (optional but keeps only the final small file)
    try:
        tmp_remap.unlink()
    except Exception as e:
        print("Warning: could not delete temp remap:", e)

    print("Saved 30 m 5-class label (HR CRS):", OUT_30M)
    print("You can align this to Landsat later on HCC with gdalwarp -r mode.")

if __name__ == "__main__":
    main()


Saved 30 m 5-class label (HR CRS): E:\Land Cover\labels_30m\2020\LabelHard_5cls_30m_2005.tif
You can align this to Landsat later on HCC with gdalwarp -r mode.


# Exporting multiple land cover labels at once 
In this step, we are only remapping and resampling the land cover since it takes a longer time without a GPU

In [3]:
#!/usr/bin/env python
# -*- coding: utf-8 -*-
"""
Batch: create compact 30 m, 5-class label rasters (remap -> downsample with mode)
in the HR raster's UTM CRS. Later, align to Landsat on HCC with gdalwarp.

Input per year:
  E:\Land Cover\exports\combinedLandcover{YYYY}.tif  (1 m categorical codes)
Output per year:
  E:\Land Cover\labels_30m\{YYYY}\LabelHard_5cls_30m_{YYYY}.tif   (uint8)
Classes:
  1=Water, 2=RiparianVeg, 3=Sand/Land, 4=Crops, 5=Other; nodata=255
"""

from pathlib import Path
import math
import numpy as np
import rasterio
from rasterio.enums import Resampling
from rasterio.transform import from_origin
from rasterio.vrt import WarpedVRT

# ---------------- CONFIG ----------------
BASE       = Path(r"E:\Land Cover")
IN_DIR     = BASE / "exports"
OUT_ROOT   = BASE / "labels_30m"
#YEARS      = [2003, 2004, 2006, 2007, 2008, 2009, 2011, 2012, 2013, 2014, 2016, 2018, 2019, 2021]   # <— edit as needed
YEARS      = [2000, 2005, 2010, 2015, 2020]   # <— edit as needed
#YEARS      = [2000]
TARGET_RES = 30.0   # meters (UTM grid)
OUT_NODATA = 255
RESAMPLE_METHOD = getattr(Resampling, "mode", Resampling.nearest)
# 1=Water, 2=RiparianVeg, 3=Sand/Land, 4=Crops,
# 5=Wetlands, 6=UplandVeg, 7=Built-up
CLASS_MAP = {
    0: 0,  # NoData stays 0

    # --- 1: Water ---
    3: 1,     # Canal/Drainage
    4: 1,     # Irrigation Reuse Pit
    5: 1,     # Lagoon
    13: 1,    # Stock Pond
    22: 1,    # River Channel
    27: 1,    # Warmwater Slough
    6000: 1,  # Water

    # --- 2: Riparian Vegetation ---
    8: 2,     # Phragmites
    9: 2,     # Purple Loosestrife
    10: 2,    # Riparian Woodland
    21: 2,    # Riparian Shrubland
    23: 2,    # River Early Successional
    24: 2,    # River Shrubland
    2000: 2,  # Veg <2 ft (riparian height class)
    3000: 2,  # Veg >15 ft
    4000: 2,  # Veg 2–6 ft
    5000: 2,  # Veg 6–15 ft

    # --- 3: Sand / Land (bare/open) ---
    2: 3,     # Bareground/Sparse Veg
    6: 3,     # Meadow Sand Ridge
    25: 3,    # Sand Pit
    26: 3,    # Unvegetated Sandbar
    1000: 3,  # Sand
    20000: 3, # Disking

    # --- 4: Crops / Farmland ---
    1: 4,     # Upland Agriculture
    28: 4,    # Agricultural Wetland (treated as ag use)
    300: 4,   # Upland Ag – Corn
    400: 4,   # Upland Ag – Soybeans
    500: 4,   # Upland Ag – Alfalfa
    600: 4,   # Upland Ag – Other Ag

    # --- 5: Wetlands ---
    7: 5,     # Mesic Wet Meadow
    20: 5,    # Floodplain Marsh
    29: 5,    # Wet Prairie (NWI)
    30: 5,    # RWB Wetland

    # --- 6: Upland Vegetation ---
    14: 6,    # Undisturbed Grassland
    15: 6,    # Upland Grassland
    16: 6,    # Upland Shrubland
    17: 6,    # Upland Woodland
    19: 6,    # Xeric Wet Meadow
    100: 6,   # Upland Ag – Forest
    200: 6,   # Upland Ag – Grassland
    10000: 6, # Tree Clearing

    # --- 7: Built-up / Impervious ---
    11: 7,    # Roads
    12: 7,    # Rural Development
    18: 7,    # Urban/Suburban
    700: 7,   # Upland Ag – Development
}



def snap_down(v, step):  return math.floor(v / step) * step
def snap_up(v, step):    return math.ceil(v / step) * step

def process_year(year:int):
    hr_in   = IN_DIR / f"combinedLandcover{year}.tif"
    out_dir = OUT_ROOT / f"{year}"
    out_dir.mkdir(parents=True, exist_ok=True)
    out_30m = out_dir / f"LabelHard_7cls_30m_{year}.tif"

    if not hr_in.exists():
        print(f"[{year}] Missing input: {hr_in}")
        return

    if out_30m.exists():
        print(f"[{year}] Exists — skipping: {out_30m}")
        return

    tmp_remap = out_dir / "tmp_remap_7cls_uint8.tif"

    # 1) Stream-remap 1 m HR → uint8 (1..5 with 255 nodata), same grid/CRS
    with rasterio.open(hr_in) as src:
        src_profile = src.profile
        src_nodata = src.nodata
        if src_nodata is None:
            # choose a nodata that won't collide with category codes
            src_nodata = -9999

        remap_prof = src_profile.copy()
        remap_prof.update(
            driver="GTiff", dtype="uint8", count=1,
            nodata=OUT_NODATA, compress="lzw", tiled=True,
            blockxsize=256, blockysize=256
        )

        with rasterio.open(tmp_remap, "w", **remap_prof) as dst:
            for _, win in src.block_windows(1):
                block = src.read(1, window=win)
                out = np.full(block.shape, OUT_NODATA, np.uint8)

                # preserve input nodata
                if src_nodata is not None:
                    out[block == src_nodata] = OUT_NODATA

                # remap known codes -> 1..5
                # (we don't touch pixels already set to 255)
                for k, v in CLASS_MAP.items():
                    mask = (block == k)
                    if mask.any():
                        out[mask] = np.uint8(v)

                dst.write(out, 1, window=win)

    # 2) Create snapped 30 m grid (same CRS), resample with 'mode'
    with rasterio.open(hr_in) as src:
        xmin, ymin, xmax, ymax = src.bounds
        dst_crs = src.crs

    xmin_s = snap_down(xmin, TARGET_RES)
    ymin_s = snap_down(ymin, TARGET_RES)
    xmax_s = snap_up(xmax, TARGET_RES)
    ymax_s = snap_up(ymax, TARGET_RES)

    width  = int(round((xmax_s - xmin_s) / TARGET_RES))
    height = int(round((ymax_s - ymin_s) / TARGET_RES))
    dst_transform = from_origin(xmin_s, ymax_s, TARGET_RES, TARGET_RES)

    with rasterio.open(tmp_remap) as src5:
        vrt_opts = dict(
            crs=dst_crs,
            transform=dst_transform,
            width=width,
            height=height,
            resampling=RESAMPLE_METHOD,
            src_nodata=OUT_NODATA
        )
        with WarpedVRT(src5, **vrt_opts) as vrt:
            prof = vrt.profile
            prof.update(
                driver="GTiff",
                dtype="uint8",
                count=1,
                nodata=OUT_NODATA,
                compress="lzw",
                tiled=True,
                blockxsize=256, blockysize=256
            )
            with rasterio.open(out_30m, "w", **prof) as dst:
                for _, win in dst.block_windows(1):
                    data = vrt.read(1, window=win).astype("uint8", copy=False)
                    dst.write(data, 1, window=win)

    # 3) Cleanup
    try:
        tmp_remap.unlink()
    except Exception as e:
        print(f"[{year}] Warning: could not delete temp remap: {e}")

    print(f"[{year}] Saved 30 m 5-class label: {out_30m}")

def main():
    print("Years:", YEARS)
    for y in YEARS:
        process_year(int(y))
    print("Done. To align each to Landsat: gdalwarp -r mode -srcnodata 255 -dstnodata 255 ...")

if __name__ == "__main__":
    main()


Years: [2000, 2005, 2010, 2015, 2020]
[2000] Saved 30 m 5-class label: E:\Land Cover\labels_30m\2000\LabelHard_7cls_30m_2000.tif
[2005] Saved 30 m 5-class label: E:\Land Cover\labels_30m\2005\LabelHard_7cls_30m_2005.tif
[2010] Saved 30 m 5-class label: E:\Land Cover\labels_30m\2010\LabelHard_7cls_30m_2010.tif
[2015] Saved 30 m 5-class label: E:\Land Cover\labels_30m\2015\LabelHard_7cls_30m_2015.tif
[2020] Saved 30 m 5-class label: E:\Land Cover\labels_30m\2020\LabelHard_7cls_30m_2020.tif
Done. To align each to Landsat: gdalwarp -r mode -srcnodata 255 -dstnodata 255 ...


In [2]:
from osgeo import gdal

GDB_PATH = r"E:\Land Cover\combinedLandcover\combinedLandcover.gdb"

def get_rat_codes(sub_path, value_col="Value"):
    """Return the set of class IDs (codes) from the RAT."""
    ds = gdal.Open(sub_path)
    if ds is None:
        raise RuntimeError(f"Could not open {sub_path}")
    band = ds.GetRasterBand(1)
    rat = band.GetDefaultRAT()
    if rat is None:
        raise RuntimeError(f"No RAT found for {sub_path}")

    col_names = [rat.GetNameOfCol(i) for i in range(rat.GetColumnCount())]
    try:
        v_idx = col_names.index(value_col)
    except ValueError:
        raise RuntimeError(f"'{value_col}' column not found. Columns = {col_names}")

    codes = set()
    for i in range(rat.GetRowCount()):
        codes.add(rat.GetValueAsInt(i, v_idx))
    return codes

# --- list subdatasets and build {year: path} mapping ---
ds = gdal.Open(GDB_PATH)
subs = ds.GetSubDatasets()

year_paths = {}
for name, desc in subs:
    # name looks like: OpenFileGDB:"...combinedLandcover.gdb":combinedLandcover2011
    layer_name = name.split(":")[-1]  # e.g. 'combinedLandcover2011'
    if not layer_name.startswith("combinedLandcover"):
        continue
    tail = layer_name.replace("combinedLandcover", "")
    # keep only those that end with a 4-digit year
    if len(tail) == 4 and tail.isdigit():
        year = int(tail)
        year_paths[year] = name

# sort by year
years = sorted(year_paths.keys())

# --- compare RAT codes across years ---
ref_year = years[0]
ref_codes = get_rat_codes(year_paths[ref_year], value_col="Value")

all_good = True
for yr in years[1:]:
    codes = get_rat_codes(year_paths[yr], value_col="Value")
    if codes != ref_codes:
        all_good = False
        print(f"⚠ Mismatch in {yr} compared to {ref_year}")
        print("  In ref but not this year:", sorted(ref_codes - codes))
        print("  In this year but not ref:", sorted(codes - ref_codes))

if all_good:
    print("✅ All years have the SAME class IDs in the RAT (Value column).")


⚠ Mismatch in 2004 compared to 2003
  In ref but not this year: []
  In this year but not ref: [20000]
⚠ Mismatch in 2005 compared to 2003
  In ref but not this year: []
  In this year but not ref: [10000, 20000]
⚠ Mismatch in 2006 compared to 2003
  In ref but not this year: []
  In this year but not ref: [10000, 20000]
⚠ Mismatch in 2007 compared to 2003
  In ref but not this year: []
  In this year but not ref: [1000, 6000, 10000, 20000]
⚠ Mismatch in 2008 compared to 2003
  In ref but not this year: []
  In this year but not ref: [1000, 6000, 10000, 20000]
⚠ Mismatch in 2009 compared to 2003
  In ref but not this year: []
  In this year but not ref: [1000, 6000, 10000, 20000]
⚠ Mismatch in 2010 compared to 2003
  In ref but not this year: []
  In this year but not ref: [1000, 6000, 10000, 20000]
⚠ Mismatch in 2011 compared to 2003
  In ref but not this year: []
  In this year but not ref: [1000, 6000, 10000]
⚠ Mismatch in 2012 compared to 2003
  In ref but not this year: []
  In th

In [5]:
def print_class_details(subdataset_path, codes_to_check, value_col="Value", name_col="lcClass"):
    from osgeo import gdal
    ds = gdal.Open(subdataset_path)
    band = ds.GetRasterBand(1)
    rat = band.GetDefaultRAT()

    col_names = [rat.GetNameOfCol(i) for i in range(rat.GetColumnCount())]
    v_idx = col_names.index(value_col)
    n_idx = col_names.index(name_col)

    for i in range(rat.GetRowCount()):
        code = rat.GetValueAsInt(i, v_idx)
        if code in codes_to_check:
            name = rat.GetValueAsString(i, n_idx)
            print(code, name)


In [6]:
print_class_details(year_paths[2015], [1000,2000,3000,4000,5000,6000,10000])


1000 Sand
2000 Veg <2ft
3000 Veg >15ft
4000 Veg 2-6ft
5000 Veg 6-15ft
6000 Water
10000 Tree Clearing


In [9]:
from osgeo import gdal

def print_code_labels(subdataset_path, codes, value_col="Value", name_col="lcClass"):
    ds = gdal.Open(subdataset_path)
    band = ds.GetRasterBand(1)
    rat = band.GetDefaultRAT()

    col_names = [rat.GetNameOfCol(i) for i in range(rat.GetColumnCount())]
    v_idx = col_names.index(value_col)
    n_idx = col_names.index(name_col)

    print(f"\n{subdataset_path}")
    for i in range(rat.GetRowCount()):
        code = rat.GetValueAsInt(i, v_idx)
        if code in codes:
            name = rat.GetValueAsString(i, n_idx)
            print(f"  {code}: {name}")


In [10]:
base = r'OpenFileGDB:E:\Land Cover\combinedLandcover\combinedLandcover.gdb:'

special_codes = [1000, 2000, 3000, 4000, 5000, 6000, 10000, 20000]
for yr in [2003, 2005, 2010, 2015, 2020]:
    sub = base + f'combinedLandcover{yr}'
    print_code_labels(sub, special_codes)



OpenFileGDB:E:\Land Cover\combinedLandcover\combinedLandcover.gdb:combinedLandcover2003

OpenFileGDB:E:\Land Cover\combinedLandcover\combinedLandcover.gdb:combinedLandcover2005
  10000: Tree Clearing
  20000: Disking

OpenFileGDB:E:\Land Cover\combinedLandcover\combinedLandcover.gdb:combinedLandcover2010
  1000: Sand
  6000: Water
  10000: Tree Clearing
  20000: Disking

OpenFileGDB:E:\Land Cover\combinedLandcover\combinedLandcover.gdb:combinedLandcover2015
  1000: Sand
  2000: Veg <2ft
  3000: Veg >15ft
  4000: Veg 2-6ft
  5000: Veg 6-15ft
  6000: Water
  10000: Tree Clearing

OpenFileGDB:E:\Land Cover\combinedLandcover\combinedLandcover.gdb:combinedLandcover2020
  1000: Sand
  2000: Veg <2ft
  3000: Veg >15ft
  4000: Veg 2-6ft
  5000: Veg 6-15ft
  6000: Water
  10000: Tree Clearing
  20000: Disking


In [12]:
from osgeo import gdal

GDB_PATH = r"E:\Land Cover\combinedLandcover\combinedLandcover.gdb"

def get_rat_dict(subdataset_path, value_col="Value", name_col="lcClass"):
    """Return {code: label} dict from RAT."""
    ds = gdal.Open(subdataset_path)
    band = ds.GetRasterBand(1)
    rat = band.GetDefaultRAT()
    if rat is None:
        raise RuntimeError(f"No RAT in {subdataset_path}")

    col_names = [rat.GetNameOfCol(i) for i in range(rat.GetColumnCount())]
    v_idx = col_names.index(value_col)
    n_idx = col_names.index(name_col)

    out = {}
    for i in range(rat.GetRowCount()):
        code = rat.GetValueAsInt(i, v_idx)
        label = rat.GetValueAsString(i, n_idx)
        out[code] = label
    return out

# ---------------------------------------------------------
# FIND ALL YEARS IN THE GDB
# ---------------------------------------------------------
ds = gdal.Open(GDB_PATH)
subs = ds.GetSubDatasets()

year_to_path = {}
for name, _ in subs:
    part = name.split(":")[-1]
    if part.startswith("combinedLandcover"):
        tail = part.replace("combinedLandcover", "")
        if tail.isdigit():
            year_to_path[int(tail)] = name

years = sorted(year_to_path.keys())
print("Found years:", years)

# ---------------------------------------------------------
# READ RAT FOR EACH YEAR
# ---------------------------------------------------------
rat_by_year = {yr: get_rat_dict(year_to_path[yr]) for yr in years}

# ---------------------------------------------------------
# COMPARE LABEL CONSISTENCY ACROSS ALL YEARS
# ---------------------------------------------------------
all_codes = sorted({c for d in rat_by_year.values() for c in d})

mismatches = []

for code in all_codes:
    labels = {}
    for yr in years:
        if code in rat_by_year[yr]:
            labels[yr] = rat_by_year[yr][code]

    # If code exists in multiple years and labels differ
    unique_labels = set(labels.values())
    if len(unique_labels) > 1:
        mismatches.append((code, labels))

# ------------------- REPORT RESULTS ----------------------

if not mismatches:
    print("\n🎉 PERFECT MATCH!")
    print("All class IDs that appear across years have the SAME label.")
else:
    print("\n⚠️ FOUND LABEL MISMATCHES:")
    for code, lbls in mismatches:
        print(f"\nCode {code}:")
        for yr, name in lbls.items():
            print(f"  {yr}: {name}")


Found years: [2003, 2004, 2005, 2006, 2007, 2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021]

🎉 PERFECT MATCH!
All class IDs that appear across years have the SAME label.


# Grabbing 5000 random training points per class for Random Forest
The UNet labels above are full 30 m rasters. For the Random Forest classification in GEE we only need point samples, not the whole raster.

This cell reads the `LabelHard_7cls_30m_{year}.tif` files, merges the 7 classes into the 5 we want (1=Water, 2=Vegetation, 3=Crops, 4=Bare, 5=Developed), keeps points on clean (non-edge) 30 m pixels, and writes 5000 random points per class per year as a shapefile. Zip each year's `.shp/.shx/.dbf/.prj` and upload to GEE as a table asset, then sample the Landsat bands at these points with `image.sampleRegions`.

In [ ]:
#!/usr/bin/env python
# -*- coding: utf-8 -*-
"""
Grab 5000 random training points per class from the 30 m label rasters.

Reads the LabelHard_7cls_30m_{year}.tif produced by the batch cell above,
merges the 7 classes into the 5 classes we want for Random Forest, keeps
points on clean (non-edge) 30 m pixels, and writes one point shapefile per
year. Each point = one 30 m pixel center, carrying a 'class' attribute.

Upload: zip the .shp/.shx/.dbf/.prj per year and upload to GEE as a table
asset. In the GEE script, sample the Landsat features at these points
(image.sampleRegions) instead of drawing training polygons.

Final RF classes:
  1=Water  2=Vegetation  3=Crops  4=Bare(sand+sandbar)  5=Developed
"""
from pathlib import Path
import numpy as np
import rasterio
from rasterio.transform import xy
import fiona

# ---------------- CONFIG ----------------
OUT_ROOT  = Path(r"E:\Land Cover") / "labels_30m"    # folder holding the 7-class 30 m tifs (from batch cell)
SAMP_DIR  = Path(r"E:\Land Cover") / "samples_30m"   # output folder for the point shapefiles
SAMP_DIR.mkdir(parents=True, exist_ok=True)

YEARS     = [2000, 2005, 2010, 2015, 2020]
N_PER_CLS = 5000
HOMOG     = True     # keep only pixels whose 3x3 neighborhood is all one class (purer labels)
SEED      = 42
OUT_NODATA = 255

# 7-class batch output -> 5-class RF scheme
#   7cls: 1 Water  2 Riparian  3 Sand/Land  4 Crops  5 Wetlands  6 Upland-veg  7 Built-up
REMAP_7_5   = {1: 1, 2: 2, 3: 4, 4: 3, 5: 2, 6: 2, 7: 5}
CLASS_NAMES = {1: "Water", 2: "Vegetation", 3: "Crops", 4: "Bare", 5: "Developed"}


def homogeneous(a5):
    """3x3 all-same-class mask (no scipy needed). Nodata in the window breaks homogeneity."""
    tmp = a5.astype(np.int16)
    tmp[a5 == OUT_NODATA] = -1
    p = np.pad(tmp, 1, constant_values=-1)
    h, w = a5.shape
    stack = np.stack([p[i:i + h, j:j + w] for i in range(3) for j in range(3)])
    return (stack.min(0) == stack.max(0)) & (a5 != OUT_NODATA)


def sample_year(year, rng):
    tif = OUT_ROOT / f"{year}" / f"LabelHard_7cls_30m_{year}.tif"
    if not tif.exists():
        print(f"[{year}] missing {tif}")
        return

    with rasterio.open(tif) as src:
        a7 = src.read(1)
        transform, crs = src.transform, src.crs

    # merge 7 -> 5 (anything not in REMAP, including nodata 0/255, stays masked)
    a5 = np.full(a7.shape, OUT_NODATA, np.uint8)
    for k, v in REMAP_7_5.items():
        a5[a7 == k] = v

    pure_mask = homogeneous(a5) if HOMOG else (a5 != OUT_NODATA)

    out = SAMP_DIR / f"train_pts_5cls_{year}.shp"
    print(f"\n[{year}] -> {out}")
    feats = []
    for c in sorted(CLASS_NAMES):
        pure = np.argwhere(pure_mask & (a5 == c))
        pool = pure
        if len(pool) < N_PER_CLS:                        # top up from non-pure pixels if short
            extra = np.argwhere((a5 == c) & ~pure_mask)
            if len(extra):
                pool = np.vstack([pure, extra])
        take = min(N_PER_CLS, len(pool))
        pick = pool[rng.choice(len(pool), take, replace=False)] if len(pool) else pool
        for r, col in pick:
            x, y = xy(transform, int(r), int(col))       # pixel center, raster CRS
            feats.append({"geometry": {"type": "Point", "coordinates": (x, y)},
                          "properties": {"class": int(c)}})
        flag = "   <-- SHORT" if take < N_PER_CLS else ""
        print(f"   {CLASS_NAMES[c]:11s} pure={len(pure):>8d}  sampled={take}{flag}")

    schema = {"geometry": "Point", "properties": {"class": "int"}}
    with fiona.open(out, "w", driver="ESRI Shapefile",
                    crs_wkt=crs.to_wkt(), schema=schema) as dst:
        dst.writerecords(feats)


rng = np.random.default_rng(SEED)
for y in YEARS:
    sample_year(y, rng)

print("\nDone. Zip .shp/.shx/.dbf/.prj per year and upload as a GEE table asset.")
print("NOTE: 2005 has no Water/Sand code in the source, so its Water and Bare points")
print("      use a different channel definition than 2010-2020 - treat those two")
print("      classes in 2005 with care (or drop them and use hand points for 2005).")


# Final label scheme: 5 classes (one vegetation class)
This is the label scheme for the Random Forest classification, added without changing the cells above.

Remaps the 1 m HR codes into 5 classes and resamples to 30 m with majority:
**1=Water, 2=Vegetation, 3=Sand/Bare, 4=Crops, 5=Developed.**

Vegetation is a single class (trees, shrubs, grass, meadow, marsh), confirmed against the PRRIP land cover methods. The codes are still grouped in the map as `tree`, `shrub`, and `herbaceous` with comments, so a woody-vs-herbaceous (or tree-vs-non-tree) split is a quick edit later — just send those groups to separate class numbers.

Note: riparian vs upland and sandbar vs other bareground are *spatial* distinctions, not spectral — separate them **after** classification using your riparian-zone and channel boundaries, exactly as the PRRIP mapping did.

In [5]:
#!/usr/bin/env python
# -*- coding: utf-8 -*-
"""
Final label scheme: 30 m, 5-class label rasters (remap -> mode resample)
in the HR raster's UTM CRS, for the Random Forest classification.

Input per year : E:\\Land Cover\\exports\\combinedLandcover{YYYY}.tif  (1 m codes)
Output per year: E:\\Land Cover\\labels_30m\\{YYYY}\\LabelHard_RF5_30m_{YYYY}.tif  (uint8)

Classes: 1=Water  2=Vegetation  3=Sand/Bare  4=Crops  5=Developed
nodata = 255

Vegetation is one class. The codes below are grouped tree / shrub / herbaceous
(confirmed against the PRRIP methods) so you can split them later if needed.
"""
from pathlib import Path
import math
import numpy as np
import rasterio
from rasterio.enums import Resampling
from rasterio.transform import from_origin
from rasterio.vrt import WarpedVRT

# ---------------- CONFIG ----------------
BASE       = Path(r"E:\PlatteRIver\Land Cover")
IN_DIR     = BASE / "exports"
OUT_ROOT   = BASE / "labels_30m"
YEARS      = [2000, 2005, 2010, 2015, 2020]
TARGET_RES = 30.0
OUT_NODATA = 255
RESAMPLE_METHOD = getattr(Resampling, "mode", Resampling.nearest)

# Original HR code -> 5 classes. Codes left out stay nodata.
# 1=Water 2=Vegetation 3=Sand/Bare 4=Crops 5=Developed
CLASS_MAP = {
    0: 0,  # background / nodata: not remapped

    # --- 1: Water ---
    3: 1,     # Canal/Drainage
    4: 1,     # Irrigation Reuse Pit
    5: 1,     # Lagoon
    13: 1,    # Stock Pond
    22: 1,    # River Channel
    6000: 1,  # Water

    # --- 2: Vegetation (trees + shrubs + herbaceous, grouped for a possible future split) ---
    # trees:
    10: 2,    # Riparian Woodland (cottonwood, elm, red cedar)
    17: 2,    # Upland Woodland
    100: 2,   # Upland Ag - Forest (shelterbelts/woodlots)
    3000: 2,  # Veg >15 ft (woodland height)
    # shrubs (willow/dogwood):
    16: 2,    # Upland Shrubland
    21: 2,    # Riparian Shrubland
    24: 2,    # River Shrubland
    4000: 2,  # Veg 2-6 ft
    5000: 2,  # Veg 6-15 ft
    # herbaceous:
    6: 2,     # Meadow Sand Ridge (grasses)
    7: 2,     # Mesic Wet Meadow
    8: 2,     # Phragmites (reed)
    9: 2,     # Purple Loosestrife (forb)
    14: 2,    # Undisturbed Grassland
    15: 2,    # Upland Grassland
    19: 2,    # Xeric Wet Meadow
    20: 2,    # Floodplain Marsh (vegetated wetland)
    23: 2,    # River Early Successional (pioneer grasses)
    27: 2,    # Warmwater Slough (vegetated backwater; move to 1 for Water)
    29: 2,    # Wet Prairie (NWI)
    30: 2,    # RWB Wetland
    200: 2,   # Upland Ag - Grassland (pasture/hay; move to 4 to treat as cropland)
    2000: 2,  # Veg <2 ft

    # --- 3: Sand / Bare ---
    2: 3,     # Bareground/Sparse Veg
    25: 3,    # Sand Pit
    26: 3,    # Unvegetated Sandbar
    1000: 3,  # Sand
    20000: 3, # Disking (bare soil)
    10000: 3, # Tree Clearing (cleared -> bare; move to 2 if regrowth)

    # --- 4: Crops / agriculture ---
    1: 4,     # Upland Agriculture
    28: 4,    # Agricultural Wetland (NWI)
    300: 4,   # Corn
    400: 4,   # Soybeans
    500: 4,   # Alfalfa
    600: 4,   # Other Ag

    # --- 5: Developed ---
    11: 5,    # Roads
    12: 5,    # Rural Development
    18: 5,    # Urban/Suburban
    700: 5,   # Upland Ag - Development
}


def snap_down(v, step): return math.floor(v / step) * step
def snap_up(v, step):   return math.ceil(v / step) * step


def process_year(year):
    hr_in   = IN_DIR / f"combinedLandcover{year}.tif"
    out_dir = OUT_ROOT / f"{year}"; out_dir.mkdir(parents=True, exist_ok=True)
    out_30m = out_dir / f"LabelHard_RF5_30m_{year}.tif"
    if not hr_in.exists():
        print(f"[{year}] missing input {hr_in}"); return
    if out_30m.exists():
        print(f"[{year}] exists, skipping {out_30m}"); return

    tmp_remap = out_dir / "tmp_remap_RF5_uint8.tif"

    # 1) stream-remap 1 m HR -> uint8 (1..5, 255 nodata), same grid/CRS
    with rasterio.open(hr_in) as src:
        prof = src.profile.copy()
        src_nodata = src.nodata if src.nodata is not None else -9999
        prof.update(driver="GTiff", dtype="uint8", count=1, nodata=OUT_NODATA,
                    compress="lzw", tiled=True, blockxsize=256, blockysize=256)
        with rasterio.open(tmp_remap, "w", **prof) as dst:
            for _, win in src.block_windows(1):
                block = src.read(1, window=win)
                out = np.full(block.shape, OUT_NODATA, np.uint8)
                for k, v in CLASS_MAP.items():
                    if v == 0:
                        continue
                    m = (block == k)
                    if m.any():
                        out[m] = np.uint8(v)
                dst.write(out, 1, window=win)

    # 2) snapped 30 m grid in same CRS, resample with mode
    with rasterio.open(hr_in) as src:
        xmin, ymin, xmax, ymax = src.bounds
        dst_crs = src.crs
    xmin_s, ymin_s = snap_down(xmin, TARGET_RES), snap_down(ymin, TARGET_RES)
    xmax_s, ymax_s = snap_up(xmax, TARGET_RES), snap_up(ymax, TARGET_RES)
    width  = int(round((xmax_s - xmin_s) / TARGET_RES))
    height = int(round((ymax_s - ymin_s) / TARGET_RES))
    dst_transform = from_origin(xmin_s, ymax_s, TARGET_RES, TARGET_RES)

    with rasterio.open(tmp_remap) as src5:
        with WarpedVRT(src5, crs=dst_crs, transform=dst_transform, width=width,
                       height=height, resampling=RESAMPLE_METHOD, src_nodata=OUT_NODATA) as vrt:
            prof = vrt.profile
            prof.update(driver="GTiff", dtype="uint8", count=1, nodata=OUT_NODATA,
                        compress="lzw", tiled=True, blockxsize=256, blockysize=256)
            with rasterio.open(out_30m, "w", **prof) as dst:
                for _, win in dst.block_windows(1):
                    dst.write(vrt.read(1, window=win).astype("uint8", copy=False), 1, window=win)

    try:
        tmp_remap.unlink()
    except Exception as e:
        print(f"[{year}] warning: could not delete temp: {e}")
    print(f"[{year}] saved {out_30m}")


for y in YEARS:
    process_year(int(y))
print("Done. 5-class labels written.")


KeyboardInterrupt: 

# Sampling 5000 points per class and saving to shapefile
Draws 5000 random points per class from the 5-class 30 m labels above, keeping points on clean (non-edge) pixels, and writes one point shapefile per year. Zip each year's `.shp/.shx/.dbf/.prj` and upload to GEE as a table asset for Random Forest training.

Classes: 1=Water 2=Vegetation 3=Sand/Bare 4=Crops 5=Developed.

In [7]:
#!/usr/bin/env python
# -*- coding: utf-8 -*-
"""
Grab 5000 random training points per class from the 5-class 30 m label rasters
(LabelHard_RF5_30m_{year}.tif) and save one point shapefile per year for GEE.

Each point = one 30 m pixel center, carrying a 'class' attribute. Points are taken
on clean (non-edge) pixels; if a class is short, it tops up from the rest.

Classes: 1=Water 2=Vegetation 3=Sand/Bare 4=Crops 5=Developed
"""
from pathlib import Path
import numpy as np
import rasterio
from rasterio.transform import xy
import fiona

# ---------------- CONFIG ----------------
OUT_ROOT  = Path(r"E:/PlatteRIver/Land Cover/") / "labels_30m"    # folder with the 5-class 30 m tifs
SAMP_DIR  = Path(r"E:/PlatteRIver/Land Cover/") / "samples_30m"   # output folder for the point shapefiles
SAMP_DIR.mkdir(parents=True, exist_ok=True)

YEARS     = [2000, 2005, 2010, 2015, 2020]
N_PER_CLS = 5000
HOMOG     = True     # keep only pixels whose 3x3 neighborhood is all one class
SEED      = 42
NODATA    = 255
CLASS_NAMES = {1: "Water", 2: "Vegetation", 3: "Sand", 4: "Crops", 5: "Developed"}


def homogeneous(a):
    """3x3 all-same-class mask (no scipy). Nodata in the window breaks homogeneity."""
    tmp = a.astype(np.int16); tmp[a == NODATA] = -1
    p = np.pad(tmp, 1, constant_values=-1); h, w = a.shape
    s = np.stack([p[i:i + h, j:j + w] for i in range(3) for j in range(3)])
    return (s.min(0) == s.max(0)) & (a != NODATA)


def sample_year(year, rng):
    tif = OUT_ROOT / f"{year}" / f"LabelHard_RF5_30m_{year}.tif"
    if not tif.exists():
        print(f"[{year}] missing {tif}"); return
    with rasterio.open(tif) as src:
        a = src.read(1); transform, crs = src.transform, src.crs

    pure = homogeneous(a) if HOMOG else (a != NODATA)
    out = SAMP_DIR / f"train_pts_RF5_{year}.shp"
    print(f"\n[{year}] -> {out}")
    feats = []
    for c in sorted(CLASS_NAMES):
        pm = np.argwhere(pure & (a == c))
        pool = pm
        if len(pool) < N_PER_CLS:                       # top up from non-pure pixels if short
            extra = np.argwhere((a == c) & ~pure)
            if len(extra):
                pool = np.vstack([pm, extra])
        take = min(N_PER_CLS, len(pool))
        pick = pool[rng.choice(len(pool), take, replace=False)] if len(pool) else pool
        for r, col in pick:
            x, y = xy(transform, int(r), int(col))      # pixel center, raster CRS
            feats.append({"geometry": {"type": "Point", "coordinates": (x, y)},
                          "properties": {"class": int(c)}})
        flag = "   <-- SHORT" if take < N_PER_CLS else ""
        print(f"   {CLASS_NAMES[c]:11s} pure={len(pm):>8d}  sampled={take}{flag}")

    schema = {"geometry": "Point", "properties": {"class": "int"}}
    with fiona.open(out, "w", driver="ESRI Shapefile",
                    crs_wkt=crs.to_wkt(), schema=schema) as dst:
        dst.writerecords(feats)


rng = np.random.default_rng(SEED)
for y in YEARS:
    sample_year(y, rng)

print("\nDone. Zip .shp/.shx/.dbf/.prj per year and upload to GEE as a table asset.")
print("2005 note: no Water/Sand code in the source - treat Water and Sand points there")
print("           with care (or drop them for 2005 and use hand points).")



[2000] -> E:\PlatteRIver\Land Cover\samples_30m\train_pts_RF5_2000.shp
   Water       pure=   15894  sampled=5000
   Vegetation  pure=  385680  sampled=5000
   Sand        pure=    3878  sampled=5000
   Crops       pure=  598240  sampled=5000
   Developed   pure=   40560  sampled=5000

[2005] -> E:\PlatteRIver\Land Cover\samples_30m\train_pts_RF5_2005.shp
   Water       pure=     122  sampled=5000
   Vegetation  pure=  486870  sampled=5000
   Sand        pure=    5605  sampled=5000
   Crops       pure=  517884  sampled=5000
   Developed   pure=   38166  sampled=5000

[2010] -> E:\PlatteRIver\Land Cover\samples_30m\train_pts_RF5_2010.shp
   Water       pure=    5886  sampled=5000
   Vegetation  pure=  418732  sampled=5000
   Sand        pure=    7252  sampled=5000
   Crops       pure=  577828  sampled=5000
   Developed   pure=   52834  sampled=5000

[2015] -> E:\PlatteRIver\Land Cover\samples_30m\train_pts_RF5_2015.shp
   Water       pure=    1479  sampled=5000
   Vegetation  pure=  39

In [2]:
!pip install fiona

   ---------------------------------------- 0.0/24.5 MB ? eta -:--:--
   ---------------------------------------- 0.3/24.5 MB ? eta -:--:--
   - -------------------------------------- 0.8/24.5 MB 2.2 MB/s eta 0:00:11
   -- ------------------------------------- 1.3/24.5 MB 2.3 MB/s eta 0:00:11
   --- ------------------------------------ 2.1/24.5 MB 2.7 MB/s eta 0:00:09
   ---- ----------------------------------- 2.9/24.5 MB 3.1 MB/s eta 0:00:07
   ------ --------------------------------- 3.9/24.5 MB 3.5 MB/s eta 0:00:06
   --------- ------------------------------ 5.5/24.5 MB 4.0 MB/s eta 0:00:05
   ----------- ---------------------------- 6.8/24.5 MB 4.4 MB/s eta 0:00:05
   -------------- ------------------------- 8.9/24.5 MB 5.0 MB/s eta 0:00:04
   ------------------ --------------------- 11.3/24.5 MB 5.7 MB/s eta 0:00:03
   ------------------------ --------------- 14.7/24.5 MB 6.7 MB/s eta 0:00:02
   ------------------------------ --------- 18.4/24.5 MB 7.7 MB/s eta 0:00:01
   -------